# KLDM Lattice-CASAL in DiffCSP++ k-space

This notebook follows `KLDM_lattice_CASAL_kspace_plan.md` in a lightweight configuration:

- **3 graphs** for the core mechanics and KLDM-coupling tests
- **10 graphs** for the GT vs random vs actual-KLDM lattice bottleneck study
- **800 reverse steps** by default in `laptop` profile
- **1000 reverse steps** in `full/deep` profile

Repo-module mapping:

- CASAL-style sampler plumbing: `src/kldmPlus/algorithm10_casal_chart.py`
- DiffCSP++ k-space basis: `src/kldmPlus/symmetry/k_basis.py`
- KLDM reverse pipeline: `src/kldmPlus/kldm.py`
- New k-space lattice-CASAL helpers: `src/kldmPlus/algorithm12_lattice_casal_kspace.py`

Most important trust gate:

> For every graph, especially SG 227 and SG 194, GT lattice -> conventional standardization -> k should satisfy `R_k(GT) ~= 0`.
>
> If that fails, do **not** trust k-space CASAL or final k-projection.


In [32]:
import dataclasses
import math
import os
import random
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

from kldmPlus.algorithm10_casal_chart import (
    Algorithm10Config,
    _decode_lattice_matrix,
    _encode_lattice_matrix,
    _project_graph_to_chart,
)
from kldmPlus.algorithm12_lattice_casal_kspace import (
    KSpaceLatticeCASALConfig,
    KSpaceLatticeCASALState,
    cell_to_k_paper,
    cell_to_l,
    k_to_l,
    kspace_lattice_casal_step,
    l_to_cell,
    l_to_k,
    project_k_to_family,
    project_l_to_k_family,
    sample_kldm_lattice_casal_kspace,
)
from kldmPlus.run_sampling_compare import SamplingCompareRunner, TEST_SPLIT, make_fixed_subset, set_seed
from kldmPlus.sample_evaluation import evaluate_csp_reconstruction
from kldmPlus.symmetry.frame_bridge import standardize_structure
from kldmPlus.symmetry.k_basis import cell_to_k, k_to_cell_matrix, space_group_k_constraint
from kldmPlus.symmetry.pcs_projection import _build_vanilla_structure
from kldmPlus.utils.time import iter_sampling_times

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "configs").exists():
    ROOT = ROOT.parent
CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)
COMPARE_CFG = CONFIG["sampling_compare"]
PROFILE = os.environ.get("KLDM_KSPACE_CASAL_PROFILE", "laptop").strip().lower()

LAPTOP_DEFAULTS = {
    "KLDM_KSPACE_GRAPH_IDS": "1,2,3",
    "KLDM_KSPACE_ORACLE_GRAPH_IDS": "1,2,3,4,5,6,7,8,9,10",
    "KLDM_KSPACE_LATTICE_ORACLE_GRAPH_IDS": "1,2,3,4,5",
    "KLDM_KSPACE_SAMPLE_SEED": str(COMPARE_CFG.get("sample_seed", 2026+2)),
    "KLDM_KSPACE_N_STEPS": "800",
    "KLDM_KSPACE_CAPTURE_SWEEP_STEPS": "200,400,600,800",
    "KLDM_KSPACE_PROJECTION_MIN_LENGTH": "0.5",
    "KLDM_KSPACE_VOLUME_RATIO_MIN": "0.25",
    "KLDM_KSPACE_VOLUME_RATIO_MAX": "4.0",
    "KLDM_KSPACE_VALIDITY_CUTOFF": str(COMPARE_CFG.get("validity_cutoff", 1.0)),
    "KLDM_KSPACE_RUN_GATE10": "1",
    "KLDM_KSPACE_SYNTHETIC_SWEEP": "rho=0.5,tau=0.025,mu=0.25; rho=2.0,tau=0.05,mu=0.25; rho=2.0,tau=0.05,mu=0.5",
    "KLDM_KSPACE_LATE_SWEEP": "rho=0.5,tau=0.025,start=0.0,mu=0.25,feedback=x,return=z; rho=0.5,tau=0.025,start=0.0,mu=0.25,feedback=z,return=z",
}

FULL_DEFAULTS = {
    **LAPTOP_DEFAULTS,
    "KLDM_KSPACE_LATTICE_ORACLE_GRAPH_IDS": "1,2,3,4,5",
    "KLDM_KSPACE_N_STEPS": "1000",
    "KLDM_KSPACE_CAPTURE_SWEEP_STEPS": "250,500,750,1000",
    "KLDM_KSPACE_SYNTHETIC_SWEEP": "rho=0.5,tau=0.025,mu=0.25; rho=1.0,tau=0.05,mu=0.25; rho=2.0,tau=0.05,mu=0.5; rho=5.0,tau=0.1,mu=0.5",
    "KLDM_KSPACE_LATE_SWEEP": "rho=0.5,tau=0.025,start=0.0,mu=0.25,feedback=x,return=z; rho=1.0,tau=0.05,start=0.0,mu=0.25,feedback=x,return=z; rho=0.5,tau=0.025,start=0.0,mu=0.25,feedback=z,return=z",
}


def profile_default(name: str) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {"full", "deep"}:
        return FULL_DEFAULTS[name]
    return LAPTOP_DEFAULTS[name]


def parse_int_list(spec: str) -> list[int]:
    return [int(v) for v in str(spec).split(",") if str(v).strip()]


In [33]:
GRAPH_IDS = parse_int_list(profile_default("KLDM_KSPACE_GRAPH_IDS"))
ORACLE_GRAPH_IDS = parse_int_list(profile_default("KLDM_KSPACE_ORACLE_GRAPH_IDS"))
LATTICE_ORACLE_GRAPH_IDS = parse_int_list(profile_default("KLDM_KSPACE_LATTICE_ORACLE_GRAPH_IDS"))
SAMPLE_SEED = int(profile_default("KLDM_KSPACE_SAMPLE_SEED"))
LATTICE_ORACLE_SEEDS = parse_int_list(os.environ.get("KLDM_KSPACE_LATTICE_ORACLE_SEEDS", ",".join(str(SAMPLE_SEED + i) for i in range(5))))
N_STEPS = int(profile_default("KLDM_KSPACE_N_STEPS"))
CAPTURE_SWEEP_STEPS = parse_int_list(profile_default("KLDM_KSPACE_CAPTURE_SWEEP_STEPS"))
VALIDITY_CUTOFF = float(profile_default("KLDM_KSPACE_VALIDITY_CUTOFF"))
PROJECTION_MIN_LENGTH = float(profile_default("KLDM_KSPACE_PROJECTION_MIN_LENGTH"))
PROJECTION_VOLUME_RATIO_MIN = float(profile_default("KLDM_KSPACE_VOLUME_RATIO_MIN"))
PROJECTION_VOLUME_RATIO_MAX = float(profile_default("KLDM_KSPACE_VOLUME_RATIO_MAX"))
RUN_GATE10 = profile_default("KLDM_KSPACE_RUN_GATE10") != "0"
SYNTHETIC_SWEEP_SPEC = profile_default("KLDM_KSPACE_SYNTHETIC_SWEEP")
LATE_SWEEP_SPEC = profile_default("KLDM_KSPACE_LATE_SWEEP")

set_seed(SAMPLE_SEED)
runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.data.csp import validate_lattice_configuration

dataset_cfg = dict(runner.config["dataset"])
model_cfg = dict(runner.config["model"])
validate_lattice_configuration(
    lattice_representation=str(dataset_cfg.get("lattice_representation", "kldm")),
    lattice_parameterization=str(model_cfg["lattice_parameterization"]),
    lattice_diffusion_type=str(model_cfg.get("lattice_diffusion_type", "VP")),
)

task = CSPTask(
    dataset_name=str(dataset_cfg["name"]),
    lattice_parameterization=str(model_cfg["lattice_parameterization"]),
    lattice_representation=str(dataset_cfg.get("lattice_representation", "kldm")),
)
root = resolve_data_root(dataset_cfg.get("root"))
dataset_full = task.fit_dataset(root=root, split=TEST_SPLIT, download=True)
main_subset_size = max(GRAPH_IDS, default=0)
oracle_subset_size = max(ORACLE_GRAPH_IDS, default=0)
lattice_oracle_subset_size = max(LATTICE_ORACLE_GRAPH_IDS, default=0)

def build_notebook_batch(subset_size: int):
    subset = make_fixed_subset(
        dataset_full,
        subset_size=max(1, int(subset_size)),
        seed=int(runner.compare_cfg["subset_seed"]),
    )
    loader = DataLoader(
        subset,
        batch_size=max(1, int(subset_size)),
        shuffle=False,
        num_workers=int(dataset_cfg.get("num_workers", 0)),
        pin_memory=bool(dataset_cfg.get("pin_memory", False)),
        collate_fn=dataset_full.collate_fn,
    )
    batch_obj = next(iter(loader)).to(runner.device)
    return batch_obj, batch_obj.ptr.tolist()

MAIN_BATCH, MAIN_PTR = build_notebook_batch(main_subset_size)
ORACLE_BATCH, ORACLE_PTR = build_notebook_batch(oracle_subset_size)
LATTICE_ORACLE_BATCH, LATTICE_ORACLE_PTR = build_notebook_batch(lattice_oracle_subset_size)
BATCHES = {"main": MAIN_BATCH, "oracle": ORACLE_BATCH, "lattice_oracle": LATTICE_ORACLE_BATCH}
PTRS = {"main": MAIN_PTR, "oracle": ORACLE_PTR, "lattice_oracle": LATTICE_ORACLE_PTR}
GRAPH_IDS = [g for g in GRAPH_IDS if 1 <= int(g) <= int(MAIN_BATCH.num_graphs)]
ORACLE_GRAPH_IDS = [g for g in ORACLE_GRAPH_IDS if 1 <= int(g) <= int(ORACLE_BATCH.num_graphs)]
LATTICE_ORACLE_GRAPH_IDS = [g for g in LATTICE_ORACLE_GRAPH_IDS if 1 <= int(g) <= int(LATTICE_ORACLE_BATCH.num_graphs)]

sampler_cfgs = {cfg["name"]: dict(cfg) for cfg in COMPARE_CFG["samplers"]}
facit_cfg = sampler_cfgs.get("FacitKLDM_PC", COMPARE_CFG["samplers"][0])
algo10_cfg_map = dict(sampler_cfgs["Algorithm10_CASAL_chart"].get("algorithm10", {}))
BASE_ALGO10 = Algorithm10Config.from_mapping(algo10_cfg_map)


def replace_supported_dataclass(obj, **updates):
    supported = {field.name for field in dataclasses.fields(obj)}
    return dataclasses.replace(obj, **{key: value for key, value in updates.items() if key in supported})

FULL_WYCKOFF_ALGO10 = replace_supported_dataclass(
    BASE_ALGO10,
    debug=False,
    oracle_wyckoff_debug=True,
    return_best_even_if_invalid=False,
    max_templates=32,
    template_eval_limit=12,
    ssvd_max_steps=8,
    ssvd_random_restarts=0,
)

print(
    f"profile={PROFILE} graphs={GRAPH_IDS} oracle_graphs={ORACLE_GRAPH_IDS} lattice_oracle_graphs={LATTICE_ORACLE_GRAPH_IDS} "
    f"main_batch_num_graphs={int(MAIN_BATCH.num_graphs)} oracle_batch_num_graphs={int(ORACLE_BATCH.num_graphs)} lattice_oracle_batch_num_graphs={int(LATTICE_ORACLE_BATCH.num_graphs)} "
    f"n_steps={N_STEPS} capture_sweep_steps={CAPTURE_SWEEP_STEPS} lattice_oracle_seeds={LATTICE_ORACLE_SEEDS}"
)

@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_lattice: torch.Tensor
    requested_sg: int
    batch_key: str

@dataclass
class KLDMState:
    f: torch.Tensor
    v: torch.Tensor
    l: torch.Tensor
    h: torch.Tensor
    step: int
    t: float
    dt: float


def graph_slice(batch_key: str, graph_idx0: int) -> tuple[int, int]:
    ptr = PTRS[str(batch_key)]
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def graph_tensors(graph_idx0: int, *, batch_key="main", source=None):
    batch_obj = BATCHES[str(batch_key)]
    start, end = graph_slice(batch_key, graph_idx0)
    if source is None:
        pos, l, h = batch_obj.pos, batch_obj.l, batch_obj.atomic_numbers
    else:
        pos, _v, l, h = source
    return {
        "pos": pos[start:end].detach().clone(),
        "l": l[graph_idx0].detach().clone().reshape(-1),
        "h": h[start:end].detach().clone().to(torch.long),
        "sg": int(torch.as_tensor(batch_obj.space_group).reshape(-1)[graph_idx0].item()),
        "num_atoms": int(end - start),
    }


def load_test_graphs(graph_ids, *, batch_key):
    out = []
    for graph_id in graph_ids:
        graph_idx0 = int(graph_id) - 1
        g = graph_tensors(graph_idx0, batch_key=batch_key)
        out.append(
            GraphCase(
                graph_id=int(graph_id),
                graph_idx0=graph_idx0,
                atomic_numbers=g["h"],
                gt_frac=g["pos"],
                gt_lattice=g["l"],
                requested_sg=g["sg"],
                batch_key=str(batch_key),
            )
        )
    return out

GRAPH_CASES = load_test_graphs(GRAPH_IDS, batch_key="main")
ORACLE_GRAPH_CASES = load_test_graphs(ORACLE_GRAPH_IDS, batch_key="oracle")
LATTICE_ORACLE_GRAPH_CASES = load_test_graphs(LATTICE_ORACLE_GRAPH_IDS, batch_key="lattice_oracle")


mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[1, 2, 3] oracle_graphs=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] lattice_oracle_graphs=[1, 2, 3, 4, 5] main_batch_num_graphs=3 oracle_batch_num_graphs=10 lattice_oracle_batch_num_graphs=5 n_steps=800 capture_sweep_steps=[200, 400, 

In [34]:
result_tables = {}
summary_rows = []
_caches = {}

TEST_DESCRIPTIONS = {
    "0_imports_and_seed": "Imports, deterministic setup, and graph loading smoke test.",
    "1_l_roundtrip": "KLDM lattice-feature round-trip on GT, random, and actual KLDM states.",
    "2_k_metric_roundtrip": "Metric-preserving cell -> k -> cell' round-trip in DiffCSP++ k-space.",
    "3_k_map_compare": "Compare code k-map and paper k-map from the invariant log-Gram definition.",
    "4_gt_k_residual": "GT lattice -> conventional standardization -> k-residual must be near zero.",
    "5_family_projection": "Direct family projection correctness in k-space for GT and KLDM lattices.",
    "6_oracle_bottleneck": "5 fixed graphs over 5 reverse seeds: direct baseline versus GT-lattice oracle headroom summary.",
    "7_one_step_k_casal": "One-step k-space CASAL sanity using the faithful x-step, z=P(x+mu), and plain dual ascent.",
    "8_synthetic_k_casal": "Synthetic multi-step dual k-space CASAL mechanics with the faithful split update.",
    "9_late_k_casal": "Faithful k-space CASAL inside the real KLDM reverse chain, with x-feedback and per-step projection.",
    "10_full_wyckoff_after_k_casal": "Does k-space lattice guidance make final full Wyckoff projection easier?",
    "11_l_ood": "Does k-space feedback push KLDM lattice features out of distribution?",
    "12_score_sensitivity": "Does KLDM lattice score behavior look unstable after k-feedback?",
    "13_feedback_stability": "x-feedback versus z-feedback stability comparison.",
}

FAIL_LABELS = {
    "K_MAP_CONVENTION_FAIL",
    "GT_K_RESIDUAL_NOT_ZERO",
    "K_PROJECTION_VOLUME_DAMAGE",
    "K_TO_L_OOD",
    "KLDM_SCORE_EXPLODES_AFTER_K_FEEDBACK",
    "K_CASAL_DUAL_EXPLOSION",
    "K_CASAL_REDUCES_CONSTRAINT_BUT_HURTS_MATCH",
    "K_PROJECTION_RESCUES_FULL_WYCKOFF",
    "K_PROJECTION_HURTS_FULL_WYCKOFF",
    "LATTICE_NOT_BOTTLENECK",
}


def dataframe_result(name, rows):
    df = pd.DataFrame(rows)
    if "PASS" not in df.columns:
        df["PASS"] = False
    if "status" not in df.columns:
        df["status"] = np.where(df["PASS"].fillna(False).astype(bool), "PASS", "FAIL")
    result_tables[name] = df
    summary_rows.append({
        "test_name": name,
        "rows": int(len(df)),
        "pass_count": int(df["PASS"].fillna(False).sum()) if "PASS" in df else 0,
        "error_count": int(df["status"].eq("ERROR").sum()) if "status" in df else 0,
    })
    return df


def error_row(test, graph, exc, category="ERROR", action="inspect exception", **extra):
    tb_lines = traceback.format_exception(type(exc), exc, exc.__traceback__)
    row = {
        "test": test,
        "graph": graph,
        "what_it_tests": TEST_DESCRIPTIONS.get(test, "see guide"),
        "PASS": False,
        "status": "ERROR",
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "traceback_tail": "".join(tb_lines[-6:]).strip(),
        "failure_category": category,
        "action_needed": action,
    }
    row.update(extra)
    return row


def print_group_pass(df, group_cols):
    group_cols = [group_cols] if isinstance(group_cols, str) else list(group_cols)
    if df.empty or any(col not in df.columns for col in group_cols) or "PASS" not in df.columns:
        print(f"group pass summary unavailable for {group_cols}")
        return
    print(df.groupby(group_cols, dropna=False)["PASS"].any())


def family_from_sg(space_group: int | None) -> str | None:
    if space_group is None:
        return None
    sg = int(space_group)
    if sg <= 2:
        return "triclinic"
    if sg <= 15:
        return "monoclinic"
    if sg <= 74:
        return "orthorhombic"
    if sg <= 142:
        return "tetragonal"
    if sg <= 194:
        return "hexagonal_trigonal"
    return "cubic"


def case_pool(case):
    if any(int(case.graph_id) == int(item.graph_id) for item in ORACLE_GRAPH_CASES):
        return ORACLE_GRAPH_CASES
    return GRAPH_CASES


def random_lattice_for_case(case):
    pool = case_pool(case)
    idx = next(i for i, item in enumerate(pool) if int(item.graph_id) == int(case.graph_id))
    other = pool[(idx + 1) % len(pool)]
    return other.gt_lattice.detach().clone().reshape(-1)



def assert_case_alignment(case, graph_payload):
    if int(graph_payload["num_atoms"]) != int(case.gt_frac.shape[0]):
        raise ValueError(f"num_atoms mismatch for graph {case.graph_id}: {graph_payload['num_atoms']} vs {int(case.gt_frac.shape[0])}")
    if tuple(torch.as_tensor(graph_payload["h"], dtype=torch.long).reshape(-1).tolist()) != tuple(case.atomic_numbers.reshape(-1).tolist()):
        raise ValueError(f"atomic_numbers mismatch for graph {case.graph_id}")
    if int(graph_payload["sg"]) != int(case.requested_sg):
        raise ValueError(f"space_group mismatch for graph {case.graph_id}: {graph_payload['sg']} vs {case.requested_sg}")

def source_specs():
    return [("gt", 0), ("random", 0), ("kldm", CAPTURE_SWEEP_STEPS[-1])]


def gram_from_cell(cell):
    return cell @ cell.T


def gram_from_l(l, num_atoms):
    return gram_from_cell(l_to_cell(l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform))


def lattice_lengths_angles_from_cell(cell):
    lengths = torch.linalg.norm(cell, dim=-1)
    def angle(u, v):
        return torch.rad2deg(torch.acos(torch.clamp(torch.dot(u, v) / (torch.linalg.norm(u) * torch.linalg.norm(v)), -1.0, 1.0)))
    angles = torch.stack([angle(cell[1], cell[2]), angle(cell[0], cell[2]), angle(cell[0], cell[1])])
    return lengths.detach().cpu().numpy(), angles.detach().cpu().numpy()


def lattice_lengths_angles_from_l(l, num_atoms):
    return lattice_lengths_angles_from_cell(l_to_cell(l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform))


def volume_from_l(l, num_atoms):
    cell = l_to_cell(l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    return float(torch.abs(torch.linalg.det(cell)).detach().item())


def k_residual_from_k(k, space_group):
    constraint = space_group_k_constraint(space_group_number=int(space_group), device=k.device, dtype=k.dtype)
    return float(torch.linalg.norm((constraint.mask * (k - constraint.target)).reshape(-1)).detach().item()), constraint


def k_residual_from_l(l, space_group, num_atoms):
    k, cell = l_to_k(l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    residual, constraint = k_residual_from_k(k, space_group)
    return residual, k, cell, constraint


In [35]:
def _native_reverse_step_full_state(full_state, times, *, batch_obj):
    with torch.no_grad():
        preds_curr = full_state["score_network"](
            t=times.now.graph,
            pos=full_state["f_t"],
            v=full_state["v_t"],
            h=full_state["a_t"],
            l=full_state["l_t"],
            node_index=full_state["node_index"],
            edge_node_index=full_state["edge_node_index"],
        )
        score_v = full_state["sampling_tdm"].reconstruct_full_reverse_velocity_score(
            t=times.now.nodes,
            v_t=full_state["v_t"],
            pred_v=preds_curr["v"],
            index=full_state["node_index"],
        )
        full_state["f_t"], full_state["v_t"] = full_state["sampling_tdm"].reverse_exp_step(
            f_t=full_state["f_t"],
            v_t=full_state["v_t"],
            score_v=score_v,
            index=full_state["node_index"],
            dt=times.dt,
        )
        full_state["l_t"] = runner.model._reverse_lattice_sampling_step(
            t=times.now.lattice,
            x_t=full_state["l_t"],
            pred=preds_curr["l"],
            dt=times.dt,
            num_atoms=batch_obj.num_atoms,
        )
    return full_state


def capture_batch_state(seed=SAMPLE_SEED, n_steps=N_STEPS, capture_step=N_STEPS, *, batch_key="main"):
    key = ("capture", str(batch_key), int(seed), int(n_steps), int(capture_step))
    if key in _caches:
        return _caches[key]
    set_seed(seed)
    batch_obj = BATCHES[str(batch_key)]
    state = runner.model._prepare_csp_sampling(
        batch=batch_obj,
        n_steps=int(n_steps),
        t_start=float(facit_cfg.get("t_start", 1.0)),
        t_final=float(facit_cfg.get("t_final", 1.0e-3)),
    )
    last_times = None
    for step_idx, times in enumerate(iter_sampling_times(batch=state["batch"], grid=state["sampling_time_grid"]), start=1):
        last_times = times
        state = _native_reverse_step_full_state(state, times, batch_obj=batch_obj)
        if step_idx >= int(capture_step):
            break
    _caches[key] = (state, last_times, int(capture_step))
    return _caches[key]


def state_for_case(case, *, source="kldm", capture_step=N_STEPS, seed=SAMPLE_SEED):
    if source == "gt":
        return KLDMState(
            f=case.gt_frac.detach().clone(),
            v=torch.zeros_like(case.gt_frac),
            l=case.gt_lattice.detach().clone().reshape(-1),
            h=case.atomic_numbers.detach().clone().to(torch.long),
            step=0,
            t=0.0,
            dt=0.0,
        )
    if source == "random":
        return KLDMState(
            f=case.gt_frac.detach().clone(),
            v=torch.zeros_like(case.gt_frac),
            l=random_lattice_for_case(case),
            h=case.atomic_numbers.detach().clone().to(torch.long),
            step=-1,
            t=0.0,
            dt=0.0,
        )
    full_state, times, step_idx = capture_batch_state(seed=seed, n_steps=N_STEPS, capture_step=capture_step, batch_key=case.batch_key)
    start, end = graph_slice(case.batch_key, case.graph_idx0)
    return KLDMState(
        f=full_state["f_t"][start:end].detach().clone(),
        v=full_state["v_t"][start:end].detach().clone(),
        l=full_state["l_t"][case.graph_idx0].detach().clone().reshape(-1),
        h=full_state["a_t"][start:end].detach().clone().to(torch.long),
        step=int(step_idx),
        t=float(times.now.graph[case.graph_idx0].detach().reshape(-1)[0].item()) if times is not None else float("nan"),
        dt=float(times.dt) if times is not None else 0.0,
    )


def vanilla_structure_for_case(case, *, frac=None, l=None, h=None):
    frac = case.gt_frac if frac is None else frac
    l = case.gt_lattice if l is None else l
    h = case.atomic_numbers if h is None else h
    cell = l_to_cell(l, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
    return _build_vanilla_structure(
        frac_coords=torch.as_tensor(frac),
        atomic_numbers=torch.as_tensor(h, dtype=torch.long),
        cell_matrix=cell,
    )


def standardized_gt_cell(case, *, standardization="conventional", symprec=1e-2, angle_tolerance=5.0):
    structure = vanilla_structure_for_case(case)
    analyzer, standardized = standardize_structure(
        structure,
        standardization=standardization,
        symprec=symprec,
        angle_tolerance=angle_tolerance,
    )
    return analyzer, standardized, torch.as_tensor(np.asarray(standardized.lattice.matrix, dtype=float), dtype=case.gt_lattice.dtype)


def evaluate_arrays(method, case, pred_f, pred_l, pred_h, **extra):
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h,
        target_f=case.gt_frac,
        target_l=case.gt_lattice,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=float(VALIDITY_CUTOFF),
    )
    k_residual, _k, _cell, _constraint = k_residual_from_l(pred_l, case.requested_sg, int(case.gt_frac.shape[0]))
    standardized_frac_rmse = None
    if result.matcher_diagnostics is not None:
        standardized_frac_rmse = result.matcher_diagnostics.standardized_frac_rmse
    row = {
        "method": method,
        "graph": case.graph_id,
        "sg": case.requested_sg,
        "match": bool(result.match),
        "rmse": np.nan if result.rmse is None else float(result.rmse),
        "frac_rmse": np.nan if result.frac_rmse is None else float(result.frac_rmse),
        "volume_ratio": volume_from_l(pred_l, int(case.gt_frac.shape[0])) / max(volume_from_l(case.gt_lattice, int(case.gt_frac.shape[0])), 1.0e-12),
        "lattice_lengths_mae": np.nan if result.lattice_lengths_mae is None else float(result.lattice_lengths_mae),
        "lattice_angles_mae": np.nan if result.lattice_angles_mae is None else float(result.lattice_angles_mae),
        "standardized_frac_rmse": np.nan if standardized_frac_rmse is None else float(standardized_frac_rmse),
        "k_residual": k_residual,
        "detected_sg": result.detected_space_group,
        "requested_sg_match": bool(result.requested_space_group_match) if result.requested_space_group_match is not None else False,
        "composition_match": bool(result.composition_match) if result.composition_match is not None else False,
        "valid": bool(result.valid),
        "validity_reason": result.validity_reason,
        "min_pair_distance": np.nan if result.min_pair_distance is None else float(result.min_pair_distance),
    }
    row.update(extra)
    return row


def parse_sweep(spec: str):
    combos = []
    for chunk in str(spec).split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        combo = {}
        for part in chunk.split(","):
            key, value = part.split("=", 1)
            key = key.strip()
            value = value.strip()
            if key in {"feedback", "return"}:
                combo[key] = value
            else:
                combo[key] = float(value)
        combos.append(combo)
    return combos


def full_wyckoff_project(case, pred_f, pred_l, pred_h, *, config_override=None):
    cfg = FULL_WYCKOFF_ALGO10 if config_override is None else config_override
    return _project_graph_to_chart(
        graph_idx=case.graph_idx0,
        requested_sg=int(case.requested_sg),
        target_pos=torch.as_tensor(pred_f).detach().clone(),
        target_l=torch.as_tensor(pred_l).detach().clone().reshape(-1),
        target_h=torch.as_tensor(pred_h).detach().clone().to(torch.long),
        lattice_transform=runner.lattice_transform,
        config=cfg,
        template_prior=None,
        oracle_reference_structure=None,
    )


## Gate 0 — Imports and graph setup

In [36]:
rows = []
try:
    for case in GRAPH_CASES:
        rows.append({
            "test": "0_imports_and_seed",
            "graph": case.graph_id,
            "what_it_tests": TEST_DESCRIPTIONS["0_imports_and_seed"],
            "requested_sg": case.requested_sg,
            "num_atoms": int(case.gt_frac.shape[0]),
            "PASS": True,
            "status": "PASS",
        })
    print(f"KSPACE_SETUP_READY={len(rows) == len(GRAPH_CASES)}")
except Exception as exc:
    rows.append(error_row("0_imports_and_seed", -1, exc, "SETUP_ERROR"))
    print("KSPACE_SETUP_READY=False")
df_0 = dataframe_result("0_imports_and_seed", rows)
display(df_0)


KSPACE_SETUP_READY=True


,test,graph,what_it_tests,requested_sg,num_atoms,PASS,status
0,0_imports_and_seed,1,"Imports, deterministic setup, and graph loadin...",227,6,True,PASS
1,0_imports_and_seed,2,"Imports, deterministic setup, and graph loadin...",4,16,True,PASS
2,0_imports_and_seed,3,"Imports, deterministic setup, and graph loadin...",194,8,True,PASS


## Gate 1 — KLDM l round trip

In [37]:
rows = []
for case in GRAPH_CASES:
    for source, capture_step in source_specs():
        try:
            state = state_for_case(case, source=source, capture_step=capture_step)
            cell0 = l_to_cell(state.l, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            l1 = cell_to_l(cell0, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            cell1 = l_to_cell(l1, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            l_error = float(torch.linalg.norm((state.l.reshape(-1) - l1.reshape(-1)).detach()).item())
            gram0 = gram_from_cell(cell0)
            gram1 = gram_from_cell(cell1)
            gram_rel = float(torch.linalg.norm((gram0 - gram1).reshape(-1)).detach().item() / max(torch.linalg.norm(gram0.reshape(-1)).detach().item(), 1.0e-12))
            lengths0, angles0 = lattice_lengths_angles_from_cell(cell0)
            lengths1, angles1 = lattice_lengths_angles_from_cell(cell1)
            length_error = float(np.sqrt(np.mean((lengths0 - lengths1) ** 2)))
            angle_error = float(np.sqrt(np.mean((angles0 - angles1) ** 2)))
            passed = bool(gram_rel < 1.0e-5 and length_error < 1.0e-5 and angle_error < 1.0e-4)
            rows.append({
                "test": "1_l_roundtrip",
                "graph": case.graph_id,
                "source": source,
                "what_it_tests": TEST_DESCRIPTIONS["1_l_roundtrip"],
                "l_error": l_error,
                "cell_metric_error": gram_rel,
                "length_error": length_error,
                "angle_error": angle_error,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "L_ROUNDTRIP_UNSTABLE",
            })
        except Exception as exc:
            rows.append(error_row("1_l_roundtrip", case.graph_id, exc, "L_ROUNDTRIP_UNSTABLE", source=source))
df_1 = dataframe_result("1_l_roundtrip", rows)
display(df_1)
print_group_pass(df_1, ["graph", "source"])


,test,graph,source,what_it_tests,l_error,cell_metric_error,length_error,angle_error,PASS,status,failure_category
0,1_l_roundtrip,1,gt,"KLDM lattice-feature round-trip on GT, random,...",1.192093e-07,4.477140e-08,0.0,0.0,True,PASS,None
1,1_l_roundtrip,1,random,"KLDM lattice-feature round-trip on GT, random,...",0.000000e+00,0.000000e+00,0.0,0.0,True,PASS,None
2,1_l_roundtrip,1,kldm,"KLDM lattice-feature round-trip on GT, random,...",1.032383e-07,0.000000e+00,0.0,0.0,True,PASS,None
3,1_l_roundtrip,2,gt,"KLDM lattice-feature round-trip on GT, random,...",0.000000e+00,0.000000e+00,0.0,0.0,True,PASS,None
4,1_l_roundtrip,2,random,"KLDM lattice-feature round-trip on GT, random,...",0.000000e+00,0.000000e+00,0.0,0.0,True,PASS,None
5,1_l_roundtrip,2,kldm,"KLDM lattice-feature round-trip on GT, random,...",5.795192e-08,0.000000e+00,0.0,0.0,True,PASS,None
6,1_l_roundtrip,3,gt,"KLDM lattice-feature round-trip on GT, random,...",0.000000e+00,0.000000e+00,0.0,0.0,True,PASS,None
7,1_l_roundtrip,3,random,"KLDM lattice-feature round-trip on GT, random,...",0.000000e+00,0.000000e+00,0.0,0.0,True,PASS,None
8,1_l_roundtrip,3,kldm,"KLDM lattice-feature round-trip on GT, random,...",1.311942e-07,0.000000e+00,0.0,0.0,True,PASS,None


graph  source
1      gt        True
       kldm      True
       random    True
2      gt        True
       kldm      True
       random    True
3      gt        True
       kldm      True
       random    True
Name: PASS, dtype: bool


## Gate 2 — k metric round trip

In [38]:
rows = []
for case in GRAPH_CASES:
    for source, capture_step in source_specs():
        try:
            state = state_for_case(case, source=source, capture_step=capture_step)
            cell0 = l_to_cell(state.l, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            gram0 = gram_from_cell(cell0)
            k = cell_to_k(cell0, eps=1.0e-8)
            cell1 = k_to_cell_matrix(k).reshape(3, 3)
            gram1 = gram_from_cell(cell1)
            raw_cell_rel_error = float(torch.linalg.norm((cell0 - cell1).reshape(-1)).detach().item() / max(torch.linalg.norm(cell0.reshape(-1)).detach().item(), 1.0e-12))
            gram_rel_error = float(torch.linalg.norm((gram0 - gram1).reshape(-1)).detach().item() / max(torch.linalg.norm(gram0.reshape(-1)).detach().item(), 1.0e-12))
            lengths0, angles0 = lattice_lengths_angles_from_cell(cell0)
            lengths1, angles1 = lattice_lengths_angles_from_cell(cell1)
            length_rmse = float(np.sqrt(np.mean((lengths0 - lengths1) ** 2)))
            angle_rmse = float(np.sqrt(np.mean((angles0 - angles1) ** 2)))
            volume_ratio = float(abs(torch.linalg.det(cell1)).detach().item() / max(abs(torch.linalg.det(cell0)).detach().item(), 1.0e-12))
            passed = bool(gram_rel_error < 1.0e-5 and length_rmse < 1.0e-5 and angle_rmse < 1.0e-4)
            rows.append({
                "test": "2_k_metric_roundtrip",
                "graph": case.graph_id,
                "source": source,
                "what_it_tests": TEST_DESCRIPTIONS["2_k_metric_roundtrip"],
                "raw_cell_rel_error": raw_cell_rel_error,
                "gram_rel_error": gram_rel_error,
                "length_rmse": length_rmse,
                "angle_rmse": angle_rmse,
                "volume_ratio": volume_ratio,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "K_MAP_CONVENTION_FAIL",
            })
        except Exception as exc:
            rows.append(error_row("2_k_metric_roundtrip", case.graph_id, exc, "K_MAP_CONVENTION_FAIL", source=source))
df_2 = dataframe_result("2_k_metric_roundtrip", rows)
display(df_2)
print_group_pass(df_2, ["graph", "source"])


,test,graph,source,what_it_tests,raw_cell_rel_error,gram_rel_error,length_rmse,angle_rmse,volume_ratio,PASS,status,failure_category
0,2_k_metric_roundtrip,1,gt,Metric-preserving cell -> k -> cell' round-tri...,3.338169e-01,1.745223e-06,3.784778e-06,0.000038,1.000001,True,PASS,None
1,2_k_metric_roundtrip,1,random,Metric-preserving cell -> k -> cell' round-tri...,5.060276e-04,6.082020e-07,1.231188e-06,0.000013,1.000000,True,PASS,None
2,2_k_metric_roundtrip,1,kldm,Metric-preserving cell -> k -> cell' round-tri...,3.201391e-01,1.264531e-06,2.952286e-06,0.000031,1.000000,True,PASS,None
3,2_k_metric_roundtrip,2,gt,Metric-preserving cell -> k -> cell' round-tri...,5.060276e-04,6.082020e-07,1.231188e-06,0.000013,1.000000,True,PASS,None
4,2_k_metric_roundtrip,2,random,Metric-preserving cell -> k -> cell' round-tri...,2.280165e-01,1.440646e-07,3.893359e-07,0.000000,1.000000,True,PASS,None
5,2_k_metric_roundtrip,2,kldm,Metric-preserving cell -> k -> cell' round-tri...,2.471280e-02,8.494842e-07,2.060172e-06,0.000018,1.000001,True,PASS,None
6,2_k_metric_roundtrip,3,gt,Metric-preserving cell -> k -> cell' round-tri...,2.280165e-01,1.440646e-07,3.893359e-07,0.000000,1.000000,True,PASS,None
7,2_k_metric_roundtrip,3,random,Metric-preserving cell -> k -> cell' round-tri...,2.979654e-07,6.360801e-07,1.557344e-06,0.000000,0.999999,True,PASS,None
8,2_k_metric_roundtrip,3,kldm,Metric-preserving cell -> k -> cell' round-tri...,2.378837e-01,8.851310e-07,1.348699e-06,0.000040,0.999999,True,PASS,None


graph  source
1      gt        True
       kldm      True
       random    True
2      gt        True
       kldm      True
       random    True
3      gt        True
       kldm      True
       random    True
Name: PASS, dtype: bool


## Gate 3 — Compare code k-map and paper k-map

In [39]:
rows = []
for case in GRAPH_CASES:
    for source, capture_step in source_specs():
        try:
            state = state_for_case(case, source=source, capture_step=capture_step)
            cell = l_to_cell(state.l, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            k_code = cell_to_k(cell, eps=1.0e-8).reshape(-1)
            k_paper = cell_to_k_paper(cell, eps=1.0e-8).reshape(-1)
            k_diff_norm = float(torch.linalg.norm((k_code - k_paper).reshape(-1)).detach().item())
            passed = bool(k_diff_norm < 1.0e-6)
            rows.append({
                "test": "3_k_map_compare",
                "graph": case.graph_id,
                "source": source,
                "what_it_tests": TEST_DESCRIPTIONS["3_k_map_compare"],
                "k_code": [float(x) for x in k_code.detach().cpu().tolist()],
                "k_paper": [float(x) for x in k_paper.detach().cpu().tolist()],
                "k_diff_norm": k_diff_norm,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "K_MAP_CONVENTION_FAIL",
            })
        except Exception as exc:
            rows.append(error_row("3_k_map_compare", case.graph_id, exc, "K_MAP_CONVENTION_FAIL", source=source))
df_3 = dataframe_result("3_k_map_compare", rows)
display(df_3[["graph", "source", "k_diff_norm", "PASS", "status"]])
print_group_pass(df_3, ["graph", "source"])


,graph,source,k_diff_norm,PASS,status
0,1,gt,0.0,True,PASS
1,1,random,0.0,True,PASS
2,1,kldm,0.0,True,PASS
3,2,gt,0.0,True,PASS
4,2,random,0.0,True,PASS
5,2,kldm,0.0,True,PASS
6,3,gt,0.0,True,PASS
7,3,random,0.0,True,PASS
8,3,kldm,0.0,True,PASS


graph  source
1      gt        True
       kldm      True
       random    True
2      gt        True
       kldm      True
       random    True
3      gt        True
       kldm      True
       random    True
Name: PASS, dtype: bool


## Gate 4 — GT conventional-frame k residual (must pass before trusting k-space CASAL)

In [40]:
rows = []
for case in ORACLE_GRAPH_CASES:
    try:
        raw_residual, raw_k, raw_cell, raw_constraint = k_residual_from_l(case.gt_lattice, case.requested_sg, int(case.gt_frac.shape[0]))
        raw_structure = vanilla_structure_for_case(case)
        analyzer, standardized_structure, standardized_cell = standardized_gt_cell(case, standardization="conventional")
        std_k = cell_to_k(standardized_cell, eps=1.0e-8).reshape(-1)
        std_residual, _constraint = k_residual_from_k(std_k, case.requested_sg)
        standardized_detected_sg = int(standardize_structure(raw_structure, standardization="conventional")[0].get_space_group_number())
        passed = bool(std_residual < 1.0e-4)
        rows.append({
            "test": "4_gt_k_residual",
            "graph": case.graph_id,
            "what_it_tests": TEST_DESCRIPTIONS["4_gt_k_residual"],
            "sg": case.requested_sg,
            "family": family_from_sg(case.requested_sg),
            "gt_k_residual_raw": raw_residual,
            "gt_k_residual_standardized": std_residual,
            "detected_sg": int(analyzer.get_space_group_number()),
            "standardized_detected_sg": standardized_detected_sg,
            "requested_sg_match": int(analyzer.get_space_group_number()) == int(case.requested_sg),
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "GT_K_RESIDUAL_NOT_ZERO",
        })
    except Exception as exc:
        rows.append(error_row("4_gt_k_residual", case.graph_id, exc, "GT_K_RESIDUAL_NOT_ZERO"))
df_4 = dataframe_result("4_gt_k_residual", rows)
display(df_4)
GT_K_CONSTRAINT_VALID = bool(df_4["PASS"].fillna(False).all()) if not df_4.empty else False
print(f"GT_K_CONSTRAINT_VALID={GT_K_CONSTRAINT_VALID}")
if not GT_K_CONSTRAINT_VALID:
    print("Do not trust k-space CASAL or final k-projection until this gate passes.")


,test,graph,what_it_tests,sg,family,gt_k_residual_raw,gt_k_residual_standardized,detected_sg,standardized_detected_sg,requested_sg_match,PASS,status,failure_category
0,4_gt_k_residual,1,GT lattice -> conventional standardization -> ...,227,cubic,4.001889e-01,9.799678e-09,227,227,True,True,PASS,None
1,4_gt_k_residual,2,GT lattice -> conventional standardization -> ...,4,monoclinic,3.635197e-08,0.000000e+00,4,4,True,True,PASS,None
2,4_gt_k_residual,3,GT lattice -> conventional standardization -> ...,194,hexagonal_trigonal,3.971742e-01,2.980232e-08,194,194,True,True,PASS,None
3,4_gt_k_residual,4,GT lattice -> conventional standardization -> ...,123,tetragonal,1.840753e-07,7.617081e-08,123,123,True,True,PASS,None
4,4_gt_k_residual,5,GT lattice -> conventional standardization -> ...,123,tetragonal,1.129852e-07,2.712914e-07,123,123,True,True,PASS,None
5,4_gt_k_residual,6,GT lattice -> conventional standardization -> ...,123,tetragonal,1.664791e-07,1.527413e-08,123,123,True,True,PASS,None
6,4_gt_k_residual,7,GT lattice -> conventional standardization -> ...,71,orthorhombic,2.699376e-01,2.118577e-08,71,71,True,True,PASS,None
7,4_gt_k_residual,8,GT lattice -> conventional standardization -> ...,193,hexagonal_trigonal,4.221215e-01,2.980232e-08,193,193,True,True,PASS,None
8,4_gt_k_residual,9,GT lattice -> conventional standardization -> ...,225,cubic,4.001889e-01,3.213055e-07,225,225,True,True,PASS,None
9,4_gt_k_residual,10,GT lattice -> conventional standardization -> ...,15,monoclinic,2.405317e-01,0.000000e+00,15,15,True,True,PASS,None


GT_K_CONSTRAINT_VALID=True


## Gate 5 — Direct family projection correctness

In [41]:
rows = []
for case in GRAPH_CASES:
    for source, capture_step in [("gt", 0), ("kldm", CAPTURE_SWEEP_STEPS[-1])]:
        try:
            state = state_for_case(case, source=source, capture_step=capture_step)
            proj = project_l_to_k_family(
                l=state.l,
                space_group=case.requested_sg,
                num_atoms=int(case.gt_frac.shape[0]),
                lattice_transform=runner.lattice_transform,
                volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
                volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
                min_length=PROJECTION_MIN_LENGTH,
            )
            lengths_after, angles_after = lattice_lengths_angles_from_cell(proj.cell_projected)
            passed = bool(proj.k_residual_after < 1.0e-6 and proj.finite_ok and proj.positive_volume_ok)
            rows.append({
                "test": "5_family_projection",
                "graph": case.graph_id,
                "source": source,
                "what_it_tests": TEST_DESCRIPTIONS["5_family_projection"],
                "k_res_before": proj.k_residual_before,
                "k_res_after": proj.k_residual_after,
                "family_after": family_from_sg(case.requested_sg),
                "volume_ratio": proj.volume_ratio_to_input,
                "lengths_after": [float(x) for x in lengths_after.tolist()],
                "angles_after": [float(x) for x in angles_after.tolist()],
                "physical_ok": proj.physical_ok,
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "K_PROJECTION_VOLUME_DAMAGE",
            })
        except Exception as exc:
            rows.append(error_row("5_family_projection", case.graph_id, exc, "K_PROJECTION_VOLUME_DAMAGE", source=source))
df_5 = dataframe_result("5_family_projection", rows)
display(df_5)


,test,graph,source,what_it_tests,k_res_before,k_res_after,family_after,volume_ratio,lengths_after,angles_after,physical_ok,PASS,status,failure_category
0,5_family_projection,1,gt,Direct family projection correctness in k-spac...,4.001889e-01,0.0,cubic,1.000000,"[4.7478556632995605, 4.7478556632995605, 4.747...","[90.0, 90.0, 90.0]",True,True,PASS,None
1,5_family_projection,1,kldm,Direct family projection correctness in k-spac...,3.918880e-01,0.0,cubic,1.000001,"[4.7708940505981445, 4.7708940505981445, 4.770...","[90.0, 90.0, 90.0]",True,True,PASS,None
2,5_family_projection,2,gt,Direct family projection correctness in k-spac...,3.635197e-08,0.0,monoclinic,0.999999,"[4.427343845367432, 5.275266170501709, 6.63175...","[90.0000228881836, 89.94203186035156, 90.0]",True,True,PASS,None
3,5_family_projection,2,kldm,Direct family projection correctness in k-spac...,2.504052e-02,0.0,monoclinic,1.000000,"[4.132734775543213, 5.513197898864746, 6.57284...","[90.0, 91.03790283203125, 90.0]",True,True,PASS,None
4,5_family_projection,3,gt,Direct family projection correctness in k-spac...,3.971742e-01,0.0,hexagonal_trigonal,1.000000,"[5.5470170974731445, 5.5470170974731445, 5.608...","[90.0, 90.0, 120.00000762939453]",True,True,PASS,None
5,5_family_projection,3,kldm,Direct family projection correctness in k-spac...,4.135452e-01,0.0,hexagonal_trigonal,1.000001,"[5.450009822845459, 5.450009822845459, 5.53810...","[90.0, 90.0, 120.00000762939453]",True,True,PASS,None


## Gate 6 — Oracle bottleneck tests on 10 graphs

In [ ]:
rows = []
for oracle_seed in LATTICE_ORACLE_SEEDS:
    set_seed(int(oracle_seed))
    baseline_sample = runner.model.sample_CSP_algorithm3(
        n_steps=N_STEPS,
        batch=LATTICE_ORACLE_BATCH,
        t_start=float(facit_cfg.get("t_start", 1.0)),
        t_final=float(facit_cfg.get("t_final", 1.0e-3)),
    )
    for case in LATTICE_ORACLE_GRAPH_CASES:
        try:
            g = graph_tensors(case.graph_idx0, batch_key=case.batch_key, source=baseline_sample)
            assert_case_alignment(case, g)
            baseline_cell = l_to_cell(g["l"], num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            gt_cell = l_to_cell(case.gt_lattice, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
            baseline_l_diff_norm = float(torch.linalg.norm((torch.as_tensor(g["l"]).reshape(-1) - case.gt_lattice.reshape(-1)).detach()).item())
            baseline_cell_fro_diff = float(torch.linalg.norm((baseline_cell - gt_cell).reshape(-1)).detach().item())
            baseline_cell_volume_ratio_to_gt = float(torch.abs(torch.linalg.det(baseline_cell)).detach().item() / max(torch.abs(torch.linalg.det(gt_cell)).detach().item(), 1.0e-12))
            variants = [
                ("baseline", g["pos"], g["l"], g["h"]),
                ("GT_lattice_oracle", g["pos"], case.gt_lattice, g["h"]),
            ]
            for method, pred_f, pred_l, pred_h in variants:
                row = evaluate_arrays(method, case, pred_f, pred_l, pred_h)
                row.update({
                    "test": "6_oracle_bottleneck",
                    "what_it_tests": TEST_DESCRIPTIONS["6_oracle_bottleneck"],
                    "PASS": True,
                    "status": "OBSERVED",
                    "aligned_case_pair": True,
                    "oracle_seed": int(oracle_seed),
                    "baseline_l_diff_norm_to_gt": baseline_l_diff_norm,
                    "baseline_cell_fro_diff_to_gt": baseline_cell_fro_diff,
                    "baseline_cell_volume_ratio_to_gt": baseline_cell_volume_ratio_to_gt,
                })
                rows.append(row)
        except Exception as exc:
            rows.append(error_row("6_oracle_bottleneck", case.graph_id, exc, "LATTICE_NOT_BOTTLENECK", oracle_seed=int(oracle_seed)))
df_6 = dataframe_result("6_oracle_bottleneck", rows)

baseline_ref = df_6[df_6["method"].eq("baseline")][[
    "graph", "oracle_seed", "rmse", "frac_rmse", "standardized_frac_rmse", "match", "requested_sg_match", "valid", "k_residual", "lattice_lengths_mae", "lattice_angles_mae", "volume_ratio", "baseline_l_diff_norm_to_gt", "baseline_cell_fro_diff_to_gt", "baseline_cell_volume_ratio_to_gt"
]].rename(columns={
    "rmse": "baseline_rmse",
    "frac_rmse": "baseline_frac_rmse",
    "match": "baseline_match",
    "requested_sg_match": "baseline_requested_sg_match",
    "valid": "baseline_valid",
    "k_residual": "baseline_k_residual",
    "standardized_frac_rmse": "baseline_standardized_frac_rmse",
    "lattice_lengths_mae": "baseline_lattice_lengths_mae",
    "lattice_angles_mae": "baseline_lattice_angles_mae",
    "volume_ratio": "baseline_volume_ratio",
    "baseline_l_diff_norm_to_gt": "baseline_l_diff_norm_to_gt",
    "baseline_cell_fro_diff_to_gt": "baseline_cell_fro_diff_to_gt",
    "baseline_cell_volume_ratio_to_gt": "baseline_cell_volume_ratio_to_gt",
})

gt_ref = df_6[df_6["method"].eq("GT_lattice_oracle")][[
    "graph", "oracle_seed", "rmse", "frac_rmse", "standardized_frac_rmse", "match", "requested_sg_match", "valid", "k_residual", "lattice_lengths_mae", "lattice_angles_mae", "volume_ratio"
]].rename(columns={
    "rmse": "gt_lattice_rmse",
    "frac_rmse": "gt_lattice_frac_rmse",
    "match": "gt_lattice_match",
    "requested_sg_match": "gt_lattice_requested_sg_match",
    "valid": "gt_lattice_valid",
    "k_residual": "gt_lattice_k_residual",
    "standardized_frac_rmse": "gt_lattice_standardized_frac_rmse",
    "lattice_lengths_mae": "gt_lattice_lengths_mae",
    "lattice_angles_mae": "gt_lattice_angles_mae",
    "volume_ratio": "gt_lattice_volume_ratio",
})

df_6_pair = baseline_ref.merge(gt_ref, on=["graph", "oracle_seed"], how="inner")
df_6_pair["delta_frac_rmse_vs_baseline"] = df_6_pair["gt_lattice_frac_rmse"] - df_6_pair["baseline_frac_rmse"]
df_6_pair["delta_rmse_vs_baseline"] = df_6_pair["gt_lattice_rmse"] - df_6_pair["baseline_rmse"]
df_6_pair["delta_match_vs_baseline"] = df_6_pair["gt_lattice_match"].astype(int) - df_6_pair["baseline_match"].astype(int)
df_6_pair["delta_requested_sg_vs_baseline"] = df_6_pair["gt_lattice_requested_sg_match"].astype(int) - df_6_pair["baseline_requested_sg_match"].astype(int)
df_6_pair["delta_valid_vs_baseline"] = df_6_pair["gt_lattice_valid"].astype(int) - df_6_pair["baseline_valid"].astype(int)
df_6_pair["delta_k_residual_vs_baseline"] = df_6_pair["gt_lattice_k_residual"] - df_6_pair["baseline_k_residual"]
df_6_pair["delta_standardized_frac_rmse_vs_baseline"] = df_6_pair["gt_lattice_standardized_frac_rmse"] - df_6_pair["baseline_standardized_frac_rmse"]
df_6_pair["delta_lattice_lengths_mae_vs_baseline"] = df_6_pair["gt_lattice_lengths_mae"] - df_6_pair["baseline_lattice_lengths_mae"]
df_6_pair["delta_lattice_angles_mae_vs_baseline"] = df_6_pair["gt_lattice_angles_mae"] - df_6_pair["baseline_lattice_angles_mae"]
df_6_pair["delta_volume_ratio_vs_baseline"] = df_6_pair["gt_lattice_volume_ratio"] - df_6_pair["baseline_volume_ratio"]

eps = 1.0e-8
df_6_pair["oracle_rescues_match"] = (~df_6_pair["baseline_match"]) & (df_6_pair["gt_lattice_match"])
df_6_pair["oracle_hurts_match"] = (df_6_pair["baseline_match"]) & (~df_6_pair["gt_lattice_match"])
df_6_pair["oracle_rescues_valid"] = (~df_6_pair["baseline_valid"]) & (df_6_pair["gt_lattice_valid"])
df_6_pair["oracle_hurts_valid"] = (df_6_pair["baseline_valid"]) & (~df_6_pair["gt_lattice_valid"])
df_6_pair["oracle_improves_rmse"] = (
    df_6_pair["baseline_rmse"].notna()
    & df_6_pair["gt_lattice_rmse"].notna()
    & (df_6_pair["delta_rmse_vs_baseline"] < -eps)
)
df_6_pair["oracle_improves_std_frac"] = (
    df_6_pair["baseline_standardized_frac_rmse"].notna()
    & df_6_pair["gt_lattice_standardized_frac_rmse"].notna()
    & (df_6_pair["delta_standardized_frac_rmse_vs_baseline"] < -eps)
)
df_6_pair["oracle_improves_constraint"] = df_6_pair["delta_k_residual_vs_baseline"] < -eps
df_6_pair["oracle_improves_csp_without_harm"] = (
    (~df_6_pair["oracle_hurts_match"])
    & (~df_6_pair["oracle_hurts_valid"])
    & (
        df_6_pair["oracle_rescues_match"]
        | df_6_pair["oracle_improves_rmse"]
        | df_6_pair["oracle_improves_std_frac"]
    )
)
df_6_pair["oracle_headroom_label"] = np.select(
    [
        df_6_pair["oracle_rescues_match"],
        df_6_pair["oracle_hurts_match"],
        df_6_pair["oracle_hurts_valid"],
        df_6_pair["oracle_improves_rmse"] & df_6_pair["oracle_improves_std_frac"],
        df_6_pair["oracle_improves_rmse"],
        df_6_pair["oracle_improves_std_frac"],
        df_6_pair["oracle_improves_constraint"],
    ],
    [
        "rescues_match",
        "hurts_match",
        "hurts_valid",
        "improves_rmse_and_stdfrac",
        "improves_rmse_only",
        "improves_stdfrac_only",
        "improves_constraint_only",
    ],
    default="no_effect",
)

def _safe_mean(frame, col):
    series = frame[col].dropna()
    return np.nan if len(series) == 0 else float(series.mean())

baseline_nonmatch = df_6_pair[~df_6_pair["baseline_match"]].copy()
seed_summary = df_6_pair.groupby("oracle_seed", dropna=False).apply(
    lambda frame: pd.Series({
        "pairs": int(len(frame)),
        "baseline_match_rate": float(frame["baseline_match"].mean()),
        "gt_lattice_match_rate": float(frame["gt_lattice_match"].mean()),
        "mean_delta_rmse_vs_baseline": _safe_mean(frame, "delta_rmse_vs_baseline"),
        "mean_delta_standardized_frac_rmse_vs_baseline": _safe_mean(frame, "delta_standardized_frac_rmse_vs_baseline"),
        "oracle_rescues_match_count": int(frame["oracle_rescues_match"].sum()),
        "oracle_hurts_match_count": int(frame["oracle_hurts_match"].sum()),
        "oracle_improves_csp_without_harm_count": int(frame["oracle_improves_csp_without_harm"].sum()),
    })
).reset_index()
graph_summary = df_6_pair.groupby("graph", dropna=False).apply(
    lambda frame: pd.Series({
        "seeds": int(len(frame)),
        "baseline_match_rate": float(frame["baseline_match"].mean()),
        "gt_lattice_match_rate": float(frame["gt_lattice_match"].mean()),
        "mean_delta_rmse_vs_baseline": _safe_mean(frame, "delta_rmse_vs_baseline"),
        "mean_delta_standardized_frac_rmse_vs_baseline": _safe_mean(frame, "delta_standardized_frac_rmse_vs_baseline"),
        "oracle_rescues_match_count": int(frame["oracle_rescues_match"].sum()),
        "oracle_hurts_match_count": int(frame["oracle_hurts_match"].sum()),
        "oracle_improves_csp_without_harm_count": int(frame["oracle_improves_csp_without_harm"].sum()),
    })
).reset_index()
headroom_counts = (
    df_6_pair.groupby("oracle_headroom_label", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["count", "oracle_headroom_label"], ascending=[False, True])
)

df_6_summary = pd.DataFrame([{
    "graphs": int(df_6_pair["graph"].nunique()),
    "oracle_seeds": int(df_6_pair["oracle_seed"].nunique()),
    "graph_seed_pairs": int(len(df_6_pair)),
    "baseline_match_rate": float(df_6_pair["baseline_match"].mean()),
    "gt_lattice_match_rate": float(df_6_pair["gt_lattice_match"].mean()),
    "baseline_valid_rate": float(df_6_pair["baseline_valid"].mean()),
    "gt_lattice_valid_rate": float(df_6_pair["gt_lattice_valid"].mean()),
    "mean_delta_frac_rmse_vs_baseline": float(df_6_pair["delta_frac_rmse_vs_baseline"].mean()),
    "median_delta_frac_rmse_vs_baseline": float(df_6_pair["delta_frac_rmse_vs_baseline"].median()),
    "mean_delta_rmse_vs_baseline": _safe_mean(df_6_pair, "delta_rmse_vs_baseline"),
    "mean_delta_match_vs_baseline": float(df_6_pair["delta_match_vs_baseline"].mean()),
    "mean_delta_requested_sg_vs_baseline": float(df_6_pair["delta_requested_sg_vs_baseline"].mean()),
    "mean_delta_valid_vs_baseline": float(df_6_pair["delta_valid_vs_baseline"].mean()),
    "mean_delta_k_residual_vs_baseline": float(df_6_pair["delta_k_residual_vs_baseline"].mean()),
    "mean_delta_standardized_frac_rmse_vs_baseline": _safe_mean(df_6_pair, "delta_standardized_frac_rmse_vs_baseline"),
    "mean_delta_lattice_lengths_mae_vs_baseline": _safe_mean(df_6_pair, "delta_lattice_lengths_mae_vs_baseline"),
    "mean_delta_lattice_angles_mae_vs_baseline": _safe_mean(df_6_pair, "delta_lattice_angles_mae_vs_baseline"),
    "mean_delta_volume_ratio_vs_baseline": _safe_mean(df_6_pair, "delta_volume_ratio_vs_baseline"),
    "mean_baseline_l_diff_norm_to_gt": float(df_6_pair["baseline_l_diff_norm_to_gt"].mean()),
    "mean_baseline_cell_fro_diff_to_gt": float(df_6_pair["baseline_cell_fro_diff_to_gt"].mean()),
    "mean_baseline_cell_volume_ratio_to_gt": float(df_6_pair["baseline_cell_volume_ratio_to_gt"].mean()),
    "oracle_rescues_match_count": int(df_6_pair["oracle_rescues_match"].sum()),
    "oracle_hurts_match_count": int(df_6_pair["oracle_hurts_match"].sum()),
    "oracle_rescues_valid_count": int(df_6_pair["oracle_rescues_valid"].sum()),
    "oracle_hurts_valid_count": int(df_6_pair["oracle_hurts_valid"].sum()),
    "oracle_improves_rmse_count": int(df_6_pair["oracle_improves_rmse"].sum()),
    "oracle_improves_std_frac_count": int(df_6_pair["oracle_improves_std_frac"].sum()),
    "oracle_improves_constraint_count": int(df_6_pair["oracle_improves_constraint"].sum()),
    "oracle_improves_csp_without_harm_count": int(df_6_pair["oracle_improves_csp_without_harm"].sum()),
    "baseline_nonmatch_graph_seed_pairs": int(len(baseline_nonmatch)),
    "baseline_nonmatch_rescued_by_gt_lattice": int(baseline_nonmatch["oracle_rescues_match"].sum()) if len(baseline_nonmatch) else 0,
    "baseline_nonmatch_mean_delta_std_frac": _safe_mean(baseline_nonmatch, "delta_standardized_frac_rmse_vs_baseline"),
    "baseline_nonmatch_mean_delta_k_residual": _safe_mean(baseline_nonmatch, "delta_k_residual_vs_baseline"),
}])

display(df_6_summary)
display(headroom_counts)
display(seed_summary.sort_values(["oracle_seed"]))
display(graph_summary.sort_values(["graph"]))
display(df_6_pair[[
    "graph",
    "oracle_seed",
    "baseline_l_diff_norm_to_gt",
    "baseline_cell_fro_diff_to_gt",
    "baseline_cell_volume_ratio_to_gt",
    "baseline_frac_rmse",
    "gt_lattice_frac_rmse",
    "delta_frac_rmse_vs_baseline",
    "baseline_rmse",
    "gt_lattice_rmse",
    "delta_rmse_vs_baseline",
    "baseline_match",
    "gt_lattice_match",
    "delta_match_vs_baseline",
    "baseline_standardized_frac_rmse",
    "gt_lattice_standardized_frac_rmse",
    "delta_standardized_frac_rmse_vs_baseline",
    "baseline_lattice_lengths_mae",
    "gt_lattice_lengths_mae",
    "delta_lattice_lengths_mae_vs_baseline",
    "baseline_lattice_angles_mae",
    "gt_lattice_angles_mae",
    "delta_lattice_angles_mae_vs_baseline",
    "baseline_k_residual",
    "gt_lattice_k_residual",
    "delta_k_residual_vs_baseline",
    "oracle_headroom_label",
]].sort_values(["graph", "oracle_seed"]))
if not baseline_nonmatch.empty:
    print("Baseline non-match subset over repeated seeds: this is the real rescue test for lattice usefulness.")
    display(baseline_nonmatch[[
        "graph",
        "oracle_seed",
        "baseline_match",
        "gt_lattice_match",
        "delta_match_vs_baseline",
        "baseline_standardized_frac_rmse",
        "gt_lattice_standardized_frac_rmse",
        "delta_standardized_frac_rmse_vs_baseline",
        "baseline_k_residual",
        "gt_lattice_k_residual",
        "delta_k_residual_vs_baseline",
        "oracle_headroom_label",
    ]].sort_values(["graph", "oracle_seed"]))
print("Note: frac_rmse is computed from raw fractional coordinates only in sample_evaluation. For lattice-sensitive comparison, use rmse/match and standardized_frac_rmse plus lattice_lengths_mae/lattice_angles_mae from sample_evaluation.")
print("Direct sanity check: baseline_l_diff_norm_to_gt and baseline_cell_fro_diff_to_gt should be > 0 for non-identical baseline lattices.")
print("If GT lattice almost never rescues non-matches or improves standardized_frac_rmse without harming match across repeated seeds, lattice-only CASAL probably has low headroom.")


## Gate 7 — One-step k-CASAL sanity

In [ ]:
rows = []
late_step = CAPTURE_SWEEP_STEPS[-1]
for case in GRAPH_CASES:
    try:
        state = state_for_case(case, source="kldm", capture_step=late_step)
        x0, _cell0 = l_to_k(state.l, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
        z0 = project_k_to_family(
            k=x0,
            space_group=case.requested_sg,
            num_atoms=int(case.gt_frac.shape[0]),
            lattice_transform=runner.lattice_transform,
            volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
            volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
            min_length=PROJECTION_MIN_LENGTH,
        )
        gen = torch.Generator(device=x0.device).manual_seed(SAMPLE_SEED + 700 + case.graph_id)
        x1_kldm = x0 + 0.05 * torch.randn(x0.shape, generator=gen, device=x0.device, dtype=x0.dtype)
        state0 = KSpaceLatticeCASALState(
            x_k=x0.detach().clone(),
            z_k=z0.k_projected.detach().clone(),
            mu_k=torch.zeros_like(x0),
            initialized=True,
            last_projection_ok=True,
            diagnostics={},
        )
        x1, z1, state1 = kspace_lattice_casal_step(
            x_k_prev=x0,
            x_k_kldm_next=x1_kldm,
            state=state0,
            space_group=case.requested_sg,
            num_atoms=int(case.gt_frac.shape[0]),
            lattice_transform=runner.lattice_transform,
            rho=1.0,
            tau=0.05,
            mu_clip=0.25,
            dual_enabled=True,
            dual_rule="plain_eta",
            mu_eta=1.0,
            projection_target_mode="x_plus_mu",
            projection_guard_mode="none",
            volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
            volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
            min_length=PROJECTION_MIN_LENGTH,
        )
        before = float(torch.linalg.norm((x0 - z0.k_projected).reshape(-1)).detach().item())
        after = float(torch.linalg.norm((x1 - z1.k_projected).reshape(-1)).detach().item())
        passed = bool(torch.isfinite(x1).all() and torch.isfinite(z1.k_projected).all() and z1.k_residual_after < 1.0e-6 and after <= before * 5.0 + 1.0e-8)
        rows.append({
            "test": "7_one_step_k_casal",
            "graph": case.graph_id,
            "what_it_tests": TEST_DESCRIPTIONS["7_one_step_k_casal"],
            "x_z_before": before,
            "x_z_after": after,
            "k_residual_z": z1.k_residual_after,
            "mu_norm": float(torch.linalg.norm(state1.mu_k.reshape(-1)).detach().item()),
            "PASS": passed,
            "status": "PASS" if passed else "FAIL",
            "failure_category": None if passed else "K_CASAL_DUAL_EXPLOSION",
        })
    except Exception as exc:
        rows.append(error_row("7_one_step_k_casal", case.graph_id, exc, "K_CASAL_DUAL_EXPLOSION"))
df_7 = dataframe_result("7_one_step_k_casal", rows)
display(df_7)


,test,graph,what_it_tests,x_z_before,x_z_after,k_residual_z,mu_norm,PASS,status,failure_category
0,7_one_step_k_casal,1,One-step k-space CASAL sanity with a fake late...,0.391888,0.391570,0.0,0.019579,True,PASS,None
1,7_one_step_k_casal,2,One-step k-space CASAL sanity with a fake late...,0.025041,0.167049,0.0,0.008352,False,FAIL,K_CASAL_DUAL_EXPLOSION
2,7_one_step_k_casal,3,One-step k-space CASAL sanity with a fake late...,0.413545,0.412373,0.0,0.020619,True,PASS,None


## Gate 8 — Synthetic multi-step dual k-space CASAL

In [ ]:
def synthetic_kspace_casal(case, *, steps, rho, tau, mu_clip):
    gt_k, _ = l_to_k(case.gt_lattice, num_atoms=int(case.gt_frac.shape[0]), lattice_transform=runner.lattice_transform)
    gen = torch.Generator(device=gt_k.device).manual_seed(SAMPLE_SEED + 800 + case.graph_id)
    x = gt_k + 0.15 * torch.randn(gt_k.shape, generator=gen, device=gt_k.device, dtype=gt_k.dtype)
    z = project_k_to_family(
        k=x,
        space_group=case.requested_sg,
        num_atoms=int(case.gt_frac.shape[0]),
        lattice_transform=runner.lattice_transform,
        volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
        volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
        min_length=PROJECTION_MIN_LENGTH,
    )
    state = KSpaceLatticeCASALState(
        x_k=x.detach().clone(),
        z_k=z.k_projected.detach().clone(),
        mu_k=torch.zeros_like(x),
        initialized=True,
        last_projection_ok=True,
        diagnostics={},
    )
    clip_hits = 0
    history = []
    for step in range(int(steps) + 1):
        history.append({
            "step": step,
            "x_z_residual": float(torch.linalg.norm((state.x_k - state.z_k).reshape(-1)).detach().item()),
            "k_residual_z": float(z.k_residual_after),
            "mu_norm": float(torch.linalg.norm(state.mu_k.reshape(-1)).detach().item()),
            "physical_ok": bool(z.physical_ok),
        })
        if step == int(steps):
            break
        x_kldm_next = state.x_k + 0.05 * torch.randn(state.x_k.shape, generator=gen, device=state.x_k.device, dtype=state.x_k.dtype)
        x_next, z, state = kspace_lattice_casal_step(
            x_k_prev=state.x_k,
            x_k_kldm_next=x_kldm_next,
            state=state,
            space_group=case.requested_sg,
            num_atoms=int(case.gt_frac.shape[0]),
            lattice_transform=runner.lattice_transform,
            rho=float(rho),
            tau=float(tau),
            mu_clip=None if float(mu_clip) <= 0 else float(mu_clip),
            dual_enabled=True,
            dual_rule="plain_eta",
            mu_eta=1.0,
            projection_target_mode="x_plus_mu",
            projection_guard_mode="none",
            volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
            volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
            min_length=PROJECTION_MIN_LENGTH,
        )
        clip_hits += int(float(state.diagnostics.get("mu_clip_fraction", 0.0)) > 0.0)
    return pd.DataFrame(history), clip_hits / max(int(steps), 1)

rows = []
for combo in parse_sweep(SYNTHETIC_SWEEP_SPEC):
    for case in GRAPH_CASES:
        try:
            hist, clip_fraction = synthetic_kspace_casal(case, steps=10, rho=float(combo["rho"]), tau=float(combo["tau"]), mu_clip=float(combo["mu"]))
            passed = bool(
                float(hist["k_residual_z"].iloc[-1]) < 1.0e-6
                and bool(np.isfinite(hist["x_z_residual"].iloc[-1]))
                and bool(np.isfinite(hist["mu_norm"].iloc[-1]))
            )
            rows.append({
                "test": "8_synthetic_k_casal",
                "graph": case.graph_id,
                "what_it_tests": TEST_DESCRIPTIONS["8_synthetic_k_casal"],
                "rho": combo["rho"],
                "tau": combo["tau"],
                "mu_clip": combo["mu"],
                "x_z_initial": float(hist["x_z_residual"].iloc[0]),
                "x_z_final": float(hist["x_z_residual"].iloc[-1]),
                "k_res_z_final": float(hist["k_residual_z"].iloc[-1]),
                "mu_norm_final": float(hist["mu_norm"].iloc[-1]),
                "mu_clip_fraction": clip_fraction,
                "finite_ok": bool(np.isfinite(hist[["x_z_residual", "k_residual_z", "mu_norm"]].to_numpy(dtype=float)).all()),
                "PASS": passed,
                "status": "PASS" if passed else "FAIL",
                "failure_category": None if passed else "K_CASAL_DUAL_EXPLOSION",
            })
        except Exception as exc:
            rows.append(error_row("8_synthetic_k_casal", case.graph_id, exc, "K_CASAL_DUAL_EXPLOSION", rho=combo.get("rho"), tau=combo.get("tau"), mu_clip=combo.get("mu")))
df_8 = dataframe_result("8_synthetic_k_casal", rows)
display(df_8)


,test,graph,what_it_tests,rho,tau,mu_clip,x_z_initial,x_z_final,k_res_z_final,mu_norm_final,mu_clip_fraction,finite_ok,PASS,status,failure_category
0,8_synthetic_k_casal,1,Synthetic multi-step dual k-space CASAL mechan...,0.5,0.025,0.25,0.327167,0.299731,0.0,0.136643,0.0,True,True,PASS,None
1,8_synthetic_k_casal,2,Synthetic multi-step dual k-space CASAL mechan...,0.5,0.025,0.25,0.169497,0.498958,0.0,0.187067,0.0,True,True,PASS,None
2,8_synthetic_k_casal,3,Synthetic multi-step dual k-space CASAL mechan...,0.5,0.025,0.25,0.373937,0.769257,0.0,0.280170,0.0,True,True,PASS,None
3,8_synthetic_k_casal,1,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.25,0.327167,0.246546,0.0,0.039546,0.0,True,True,PASS,None
4,8_synthetic_k_casal,2,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.25,0.169497,0.254301,0.0,0.060451,0.0,True,True,PASS,None
5,8_synthetic_k_casal,3,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.25,0.373937,0.415323,0.0,0.089490,0.0,True,True,PASS,None
6,8_synthetic_k_casal,1,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.50,0.327167,0.246546,0.0,0.039546,0.0,True,True,PASS,None
7,8_synthetic_k_casal,2,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.50,0.169497,0.254301,0.0,0.060451,0.0,True,True,PASS,None
8,8_synthetic_k_casal,3,Synthetic multi-step dual k-space CASAL mechan...,2.0,0.050,0.50,0.373937,0.415323,0.0,0.089490,0.0,True,True,PASS,None


## Gate 9 — Late k-space CASAL inside KLDM

In [ ]:
set_seed(SAMPLE_SEED)
baseline = runner.model.sample_CSP_algorithm3(
    n_steps=N_STEPS,
    batch=MAIN_BATCH,
    t_start=float(facit_cfg.get("t_start", 1.0)),
    t_final=float(facit_cfg.get("t_final", 1.0e-3)),
)
rows = []
late_results = {}
for combo_id, combo in enumerate(parse_sweep(LATE_SWEEP_SPEC), start=1):
    cfg = KSpaceLatticeCASALConfig(
        rho_start=float(combo["rho"]),
        rho_end=float(combo["rho"]),
        tau_scale=float(combo["tau"]),
        mu_eta=1.0,
        dual_rule="plain_eta",
        mu_clip=float(combo["mu"]),
        dual_enabled=True,
        projection_start_fraction=float(combo["start"]),
        projection_start_step=1,
        projection_interval=1,
        projection_target_mode="x_plus_mu",
        projection_guard_mode="none",
        initialize_from_x0=True,
        projection_min_lattice_length=PROJECTION_MIN_LENGTH,
        hard_volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
        hard_volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
        feedback_mode=str(combo["feedback"]),
        return_mode=str(combo["return"]),
        debug=False,
    )
    try:
        pos_c, v_c, l_c, h_c, diag = sample_kldm_lattice_casal_kspace(
            model=runner.model,
            batch=MAIN_BATCH,
            n_steps=N_STEPS,
            lattice_transform=runner.lattice_transform,
            t_start=float(facit_cfg.get("t_start", 1.0)),
            t_final=float(facit_cfg.get("t_final", 1.0e-3)),
            config=cfg,
            return_diagnostics=True,
        )
        late_results[(combo_id, str(combo["feedback"]))] = (pos_c, l_c, h_c, diag, cfg)
        diag_by_graph = {int(item.get("graph_idx", -1)): item for item in diag}
        for case in GRAPH_CASES:
            start, end = graph_slice(case.batch_key, case.graph_idx0)
            raw = graph_tensors(case.graph_idx0, batch_key=case.batch_key, source=baseline)
            baseline_row = evaluate_arrays("baseline", case, raw["pos"], raw["l"], raw["h"])
            final_k_proj = project_l_to_k_family(
                l=raw["l"],
                space_group=case.requested_sg,
                num_atoms=int(case.gt_frac.shape[0]),
                lattice_transform=runner.lattice_transform,
                volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
                volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
                min_length=PROJECTION_MIN_LENGTH,
            )
            final_proj_row = evaluate_arrays("final_k_projection", case, raw["pos"], final_k_proj.l_projected, raw["h"])
            casal_row = evaluate_arrays(f"late_k_casal_{combo['feedback']}", case, pos_c[start:end], l_c[case.graph_idx0], h_c[start:end], feedback_mode=combo["feedback"])
            k_residual_final, _k, _cell, _constraint = k_residual_from_l(l_c[case.graph_idx0], case.requested_sg, int(case.gt_frac.shape[0]))
            d = diag_by_graph.get(case.graph_idx0, {})
            pass_constraint = bool(k_residual_final < 1.0e-6 and PROJECTION_VOLUME_RATIO_MIN <= float(casal_row["volume_ratio"]) <= PROJECTION_VOLUME_RATIO_MAX)
            pass_csp = bool(
                (bool(casal_row.get("match", False)) >= bool(baseline_row.get("match", False)))
                and (
                    pd.isna(casal_row.get("rmse")) or pd.isna(baseline_row.get("rmse")) or float(casal_row.get("rmse")) <= float(baseline_row.get("rmse")) + 1.0e-8
                )
            )
            rows.extend([
                {"test": "9_late_k_casal", "what_it_tests": TEST_DESCRIPTIONS["9_late_k_casal"], **baseline_row, "method": "baseline", "combo_id": 0, "feedback_mode": "baseline", "PASS_constraint": np.nan, "PASS_csp": np.nan, "PASS": True, "status": "BASELINE"},
                {"test": "9_late_k_casal", "what_it_tests": TEST_DESCRIPTIONS["9_late_k_casal"], **final_proj_row, "method": "final_k_projection", "combo_id": 0, "feedback_mode": "projection", "k_residual_final": final_k_proj.k_residual_after, "x_z_residual_final": np.nan, "mu_norm_final": np.nan, "PASS_constraint": bool(final_k_proj.k_residual_after < 1.0e-6 and final_k_proj.physical_ok), "PASS_csp": bool((bool(final_proj_row.get("match", False)) >= bool(baseline_row.get("match", False))) and (pd.isna(final_proj_row.get("rmse")) or pd.isna(baseline_row.get("rmse")) or float(final_proj_row.get("rmse")) <= float(baseline_row.get("rmse")) + 1.0e-8)), "PASS": True, "status": "OBSERVED"},
                {"test": "9_late_k_casal", "what_it_tests": TEST_DESCRIPTIONS["9_late_k_casal"], **casal_row, "method": f"late_k_casal_{combo['feedback']}", "combo_id": combo_id, "rho": combo["rho"], "tau_scale": combo["tau"], "mu_clip": combo["mu"], "start_fraction": combo["start"], "feedback_mode": combo["feedback"], "return_mode": combo["return"], "dual_rule": d.get("dual_rule", "plain_eta"), "projection_target_mode": d.get("projection_target_mode", "x_plus_mu"), "projection_guard_mode": d.get("projection_guard_mode", "none"), "k_residual_final": k_residual_final, "x_z_residual_final": d.get("x_z_residual_last", np.nan), "mu_norm_final": d.get("mu_norm_last", np.nan), "projection_success": d.get("num_projection_failures", 0) == 0, "PASS_constraint": pass_constraint, "PASS_csp": pass_csp, "PASS": bool(pass_constraint), "status": "PASS" if pass_constraint else "DIAG"},
            ])
    except Exception as exc:
        for case in GRAPH_CASES:
            rows.append(error_row("9_late_k_casal", case.graph_id, exc, "K_CASAL_REDUCES_CONSTRAINT_BUT_HURTS_MATCH", combo_id=combo_id, feedback_mode=combo.get("feedback")))
df_9 = dataframe_result("9_late_k_casal", rows)
display(df_9)


,test,what_it_tests,method,graph,sg,match,rmse,frac_rmse,volume_ratio,lattice_lengths_mae,...,status,k_residual_final,x_z_residual_final,mu_norm_final,rho,tau_scale,mu_clip,start_fraction,return_mode,projection_success
0,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,baseline,1,227,True,0.002512,0.236549,1.014628,0.099016,...,BASELINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,final_k_projection,1,227,False,NaN,0.236549,1.014629,0.558393,...,OBSERVED,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,late_k_casal_x,1,227,False,NaN,0.281385,0.670033,1.174685,...,PASS,5.000239e-07,0.415635,0.000686,0.5,0.025,0.25,0.9,z,True
3,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,baseline,2,4,True,0.442854,0.197683,0.966734,0.197128,...,BASELINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,final_k_projection,2,4,True,0.441891,0.197683,0.966735,0.197147,...,OBSERVED,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,late_k_casal_x,2,4,False,NaN,0.185704,1.095762,0.661115,...,PASS,1.848118e-07,0.092309,0.000153,0.5,0.025,0.25,0.9,z,True
6,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,baseline,3,194,True,0.326332,0.213000,0.953218,0.085106,...,BASELINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,final_k_projection,3,194,False,NaN,0.213000,0.953218,0.588023,...,OBSERVED,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,late_k_casal_x,3,194,True,0.412719,0.203011,1.005439,1.254640,...,PASS,1.159948e-07,0.418237,0.000594,0.5,0.025,0.25,0.9,z,True
9,9_late_k_casal,Late k-space CASAL inside the real KLDM revers...,baseline,1,227,True,0.002512,0.236549,1.014628,0.099016,...,BASELINE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Gate 10 — Full Wyckoff after k-CASAL

In [ ]:
if not RUN_GATE10:
    df_10 = dataframe_result("10_full_wyckoff_after_k_casal", [{"test": "10_full_wyckoff_after_k_casal", "graph": case.graph_id, "what_it_tests": TEST_DESCRIPTIONS["10_full_wyckoff_after_k_casal"], "PASS": False, "status": "SKIP"} for case in GRAPH_CASES])
    display(df_10)
else:
    rows = []
    set_seed(SAMPLE_SEED)
    baseline = runner.model.sample_CSP_algorithm3(
        n_steps=N_STEPS,
        batch=MAIN_BATCH,
        t_start=float(facit_cfg.get("t_start", 1.0)),
        t_final=float(facit_cfg.get("t_final", 1.0e-3)),
    )
    x_feedback_key = next(((combo_id, key) for combo_id, key in late_results if key == "x"), None)
    if x_feedback_key is None:
        raise RuntimeError("Gate 10 expects an x-feedback k-space CASAL run from Gate 9.")
    pos_c, l_c, h_c, diag, cfg = late_results[x_feedback_key]
    for case in GRAPH_CASES:
        try:
            start, end = graph_slice(case.batch_key, case.graph_idx0)
            raw = graph_tensors(case.graph_idx0, batch_key=case.batch_key, source=baseline)
            raw_proj = full_wyckoff_project(case, raw["pos"], raw["l"], raw["h"])
            raw_row = evaluate_arrays("KLDM_plus_full_Wyckoff", case, raw_proj.pos, raw_proj.l, raw_proj.h)
            final_k_proj = project_l_to_k_family(
                l=raw["l"],
                space_group=case.requested_sg,
                num_atoms=int(case.gt_frac.shape[0]),
                lattice_transform=runner.lattice_transform,
                volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
                volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
                min_length=PROJECTION_MIN_LENGTH,
            )
            k_full = full_wyckoff_project(case, raw["pos"], final_k_proj.l_projected, raw["h"])
            k_row = evaluate_arrays("KLDM_plus_final_k_projection_plus_full_Wyckoff", case, k_full.pos, k_full.l, k_full.h)
            casal_full = full_wyckoff_project(case, pos_c[start:end], l_c[case.graph_idx0], h_c[start:end])
            casal_row = evaluate_arrays("LateKSpaceCASAL_plus_full_Wyckoff", case, casal_full.pos, casal_full.l, casal_full.h)
            methods = [
                ("KLDM_plus_full_Wyckoff", raw_row, raw_proj.objective, raw_proj.l),
                ("KLDM_plus_final_k_projection_plus_full_Wyckoff", k_row, k_full.objective, k_full.l),
                ("LateKSpaceCASAL_plus_full_Wyckoff", casal_row, casal_full.objective, casal_full.l),
            ]
            baseline_match = bool(raw_row.get("match", False))
            baseline_rmse = raw_row.get("rmse", np.nan)
            for method, row, proj_obj, eval_l in methods:
                k_residual, _k, _cell, _constraint = k_residual_from_l(eval_l, case.requested_sg, int(case.gt_frac.shape[0]))
                pass_constraint = bool(
                    row.get("valid", False)
                    and k_residual < 1.0e-6
                    and PROJECTION_VOLUME_RATIO_MIN <= float(row.get("volume_ratio", np.nan)) <= PROJECTION_VOLUME_RATIO_MAX
                )
                pass_csp = bool(
                    (bool(row.get("match", False)) >= baseline_match)
                    and (pd.isna(row.get("rmse")) or pd.isna(baseline_rmse) or float(row.get("rmse")) <= float(baseline_rmse) + 1.0e-8)
                )
                rows.append({
                    "test": "10_full_wyckoff_after_k_casal",
                    "what_it_tests": TEST_DESCRIPTIONS["10_full_wyckoff_after_k_casal"],
                    **row,
                    "method": method,
                    "projection_objective": float(proj_obj),
                    "k_residual": k_residual,
                    "PASS_constraint": pass_constraint,
                    "PASS_csp": pass_csp,
                    "PASS": bool(pass_constraint),
                    "status": "PASS" if pass_constraint else "DIAG",
                })
        except Exception as exc:
            rows.append(error_row("10_full_wyckoff_after_k_casal", case.graph_id, exc, "K_PROJECTION_HURTS_FULL_WYCKOFF"))
    df_10 = dataframe_result("10_full_wyckoff_after_k_casal", rows)
    display(df_10)


KeyboardInterrupt: 

## Gate 11 — l-distribution OOD after k-CASAL

In [ ]:
rows = []
for case in GRAPH_CASES:
    try:
        raw = graph_tensors(case.graph_idx0, batch_key=case.batch_key, source=baseline)
        variants = [("baseline", raw["l"])]
        final_k_proj = project_l_to_k_family(
            l=raw["l"],
            space_group=case.requested_sg,
            num_atoms=int(case.gt_frac.shape[0]),
            lattice_transform=runner.lattice_transform,
            volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
            volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
            min_length=PROJECTION_MIN_LENGTH,
        )
        variants.append(("final_k_projection", final_k_proj.l_projected))
        for (combo_id, feedback_mode), (_pos, l_out, _h, _diag, _cfg) in late_results.items():
            if case.graph_idx0 < l_out.shape[0]:
                variants.append((f"late_k_casal_{feedback_mode}", l_out[case.graph_idx0]))
        for method, l_vec in variants:
            lengths, angles = lattice_lengths_angles_from_l(l_vec, int(case.gt_frac.shape[0]))
            rows.append({
                "test": "11_l_ood",
                "graph": case.graph_id,
                "what_it_tests": TEST_DESCRIPTIONS["11_l_ood"],
                "method": method,
                "l_norm": float(torch.linalg.norm(torch.as_tensor(l_vec).reshape(-1)).detach().item()),
                "l_mean": float(torch.as_tensor(l_vec).reshape(-1).mean().detach().item()),
                "l_std": float(torch.as_tensor(l_vec).reshape(-1).std(unbiased=False).detach().item()),
                "lengths": [float(x) for x in lengths.tolist()],
                "angles": [float(x) for x in angles.tolist()],
                "volume": volume_from_l(l_vec, int(case.gt_frac.shape[0])),
                "PASS": True,
                "status": "OBSERVED",
            })
    except Exception as exc:
        rows.append(error_row("11_l_ood", case.graph_id, exc, "K_TO_L_OOD"))
df_11 = dataframe_result("11_l_ood", rows)
display(df_11)


,test,graph,what_it_tests,method,l_norm,l_mean,l_std,lengths,angles,volume,PASS,status
0,11_l_ood,1,Does k-space feedback push KLDM lattice featur...,baseline,3.055047,0.556661,1.116101,"[5.473787307739258, 5.216038703918457, 5.28998...","[59.446353912353516, 60.64631652832031, 62.307...",108.592323,True,OBSERVED
1,11_l_ood,1,Does k-space feedback push KLDM lattice featur...,final_k_projection,2.706388,0.781267,0.781267,"[4.7708940505981445, 4.7708940505981445, 4.770...","[90.0, 90.0, 90.0]",108.592377,True,OBSERVED
2,11_l_ood,1,Does k-space feedback push KLDM lattice featur...,late_k_casal_x,2.466815,0.712108,0.712108,"[4.15460205078125, 4.15460205078125, 4.1546020...","[90.0, 90.0, 90.0]",71.711411,True,OBSERVED
3,11_l_ood,1,Does k-space feedback push KLDM lattice featur...,late_k_casal_z,1.260456,0.363862,0.363862,"[2.070363998413086, 2.070363998413086, 2.07036...","[90.0, 90.0, 90.0]",8.874423,True,OBSERVED
4,11_l_ood,2,Does k-space feedback push KLDM lattice featur...,baseline,2.912122,0.838117,0.843189,"[4.134337902069092, 5.516603946685791, 6.57470...","[92.05463409423828, 91.07561492919922, 87.9230...",149.734909,True,OBSERVED
5,11_l_ood,2,Does k-space feedback push KLDM lattice featur...,final_k_projection,2.910936,0.837858,0.842765,"[4.132734775543213, 5.513197898864746, 6.57284...","[90.0, 91.03790283203125, 90.0]",149.735062,True,OBSERVED
6,11_l_ood,2,Does k-space feedback push KLDM lattice featur...,late_k_casal_x,3.005076,0.860442,0.874483,"[3.9128458499908447, 5.5319390296936035, 7.843...","[90.00001525878906, 91.61001586914062, 90.0]",169.719788,True,OBSERVED
7,11_l_ood,2,Does k-space feedback push KLDM lattice featur...,late_k_casal_z,3.022919,0.783264,0.953679,"[6.2189812660217285, 4.2375264167785645, 6.517...","[90.00001525878906, 65.94213104248047, 90.0000...",156.827972,True,OBSERVED
8,11_l_ood,3,Does k-space feedback push KLDM lattice featur...,baseline,3.018326,0.941238,0.795269,"[4.768411636352539, 5.8390727043151855, 5.9761...","[120.89221954345703, 87.2973403930664, 88.9665...",142.457764,True,OBSERVED
9,11_l_ood,3,Does k-space feedback push KLDM lattice featur...,final_k_projection,3.002219,0.946706,0.778439,"[5.450009822845459, 5.450009822845459, 5.53810...","[90.0, 90.0, 120.00000762939453]",142.457748,True,OBSERVED


## Gate 12 — KLDM score sensitivity after k-feedback

In [ ]:
late_step = CAPTURE_SWEEP_STEPS[-1]
full_state, times, step_idx = capture_batch_state(seed=SAMPLE_SEED, n_steps=N_STEPS, capture_step=late_step, batch_key="main")
rows = []
for case in GRAPH_CASES:
    try:
        graph_idx = case.graph_idx0
        base_l = full_state["l_t"][graph_idx].detach().clone()
        variants = [("baseline", base_l)]
        final_k_proj = project_l_to_k_family(
            l=base_l,
            space_group=case.requested_sg,
            num_atoms=int(case.gt_frac.shape[0]),
            lattice_transform=runner.lattice_transform,
            volume_ratio_min=PROJECTION_VOLUME_RATIO_MIN,
            volume_ratio_max=PROJECTION_VOLUME_RATIO_MAX,
            min_length=PROJECTION_MIN_LENGTH,
        )
        variants.append(("final_k_projection", final_k_proj.l_projected))
        for (combo_id, feedback_mode), (_pos, l_out, _h, _diag, _cfg) in late_results.items():
            variants.append((f"late_k_casal_{feedback_mode}", l_out[graph_idx].detach().clone()))
        for method, l_variant in variants:
            probe_state = {key: value for key, value in full_state.items()}
            probe_state["l_t"] = full_state["l_t"].detach().clone()
            probe_state["l_t"][graph_idx] = torch.as_tensor(l_variant).reshape_as(probe_state["l_t"][graph_idx])
            preds_curr = probe_state["score_network"](
                t=times.now.graph,
                pos=probe_state["f_t"],
                v=probe_state["v_t"],
                h=probe_state["a_t"],
                l=probe_state["l_t"],
                node_index=probe_state["node_index"],
                edge_node_index=probe_state["edge_node_index"],
            )
            pred_l = preds_curr["l"][graph_idx].detach().clone()
            reverse_next = runner.model._reverse_lattice_sampling_step(
                t=times.now.lattice,
                x_t=probe_state["l_t"],
                pred=preds_curr["l"],
                dt=times.dt,
                num_atoms=MAIN_BATCH.num_atoms,
            )[graph_idx].detach().clone()
            rows.append({
                "test": "12_score_sensitivity",
                "graph": case.graph_id,
                "what_it_tests": TEST_DESCRIPTIONS["12_score_sensitivity"],
                "method": method,
                "pred_l_norm": float(torch.linalg.norm(pred_l.reshape(-1)).detach().item()),
                "reverse_step_delta_norm": float(torch.linalg.norm((reverse_next - probe_state["l_t"][graph_idx]).reshape(-1)).detach().item()),
                "finite_ok": bool(torch.isfinite(pred_l).all() and torch.isfinite(reverse_next).all()),
                "PASS": bool(torch.isfinite(pred_l).all() and torch.isfinite(reverse_next).all()),
                "status": "PASS" if bool(torch.isfinite(pred_l).all() and torch.isfinite(reverse_next).all()) else "FAIL",
                "failure_category": None if bool(torch.isfinite(pred_l).all() and torch.isfinite(reverse_next).all()) else "KLDM_SCORE_EXPLODES_AFTER_K_FEEDBACK",
            })
    except Exception as exc:
        rows.append(error_row("12_score_sensitivity", case.graph_id, exc, "KLDM_SCORE_EXPLODES_AFTER_K_FEEDBACK"))
df_12 = dataframe_result("12_score_sensitivity", rows)
display(df_12)


,test,graph,what_it_tests,method,pred_l_norm,reverse_step_delta_norm,finite_ok,PASS,status,failure_category
0,12_score_sensitivity,1,Does KLDM lattice score behavior look unstable...,baseline,2.538661,0.054997,True,True,PASS,None
1,12_score_sensitivity,1,Does KLDM lattice score behavior look unstable...,final_k_projection,13.480836,0.176449,True,True,PASS,None
2,12_score_sensitivity,1,Does KLDM lattice score behavior look unstable...,late_k_casal_x,13.408659,0.157577,True,True,PASS,None
3,12_score_sensitivity,1,Does KLDM lattice score behavior look unstable...,late_k_casal_z,14.445692,0.167862,True,True,PASS,None
4,12_score_sensitivity,2,Does KLDM lattice score behavior look unstable...,baseline,1.431209,0.040242,True,True,PASS,None
5,12_score_sensitivity,2,Does KLDM lattice score behavior look unstable...,final_k_projection,1.825363,0.044796,True,True,PASS,None
6,12_score_sensitivity,2,Does KLDM lattice score behavior look unstable...,late_k_casal_x,4.937613,0.070625,True,True,PASS,None
7,12_score_sensitivity,2,Does KLDM lattice score behavior look unstable...,late_k_casal_z,15.730186,0.178975,True,True,PASS,None
8,12_score_sensitivity,3,Does KLDM lattice score behavior look unstable...,baseline,3.054788,0.071878,True,True,PASS,None
9,12_score_sensitivity,3,Does KLDM lattice score behavior look unstable...,final_k_projection,14.832872,0.171422,True,True,PASS,None


## Gate 13 — x-feedback vs z-feedback stability

In [ ]:
rows = []
for case in GRAPH_CASES:
    try:
        x_rows = df_9[(df_9["graph"].eq(case.graph_id)) & (df_9["method"].astype(str).eq("late_k_casal_x"))]
        z_rows = df_9[(df_9["graph"].eq(case.graph_id)) & (df_9["method"].astype(str).eq("late_k_casal_z"))]
        if x_rows.empty or z_rows.empty:
            continue
        x_best = x_rows.sort_values(["PASS_constraint", "frac_rmse"], ascending=[False, True]).iloc[0]
        z_best = z_rows.sort_values(["PASS_constraint", "frac_rmse"], ascending=[False, True]).iloc[0]
        rows.append({
            "test": "13_feedback_stability",
            "graph": case.graph_id,
            "what_it_tests": TEST_DESCRIPTIONS["13_feedback_stability"],
            "x_frac_rmse": x_best.get("frac_rmse", np.nan),
            "z_frac_rmse": z_best.get("frac_rmse", np.nan),
            "x_k_residual": x_best.get("k_residual_final", np.nan),
            "z_k_residual": z_best.get("k_residual_final", np.nan),
            "x_xz_residual": x_best.get("x_z_residual_final", np.nan),
            "z_xz_residual": z_best.get("x_z_residual_final", np.nan),
            "x_volume_ratio": x_best.get("volume_ratio", np.nan),
            "z_volume_ratio": z_best.get("volume_ratio", np.nan),
            "PASS": bool(bool(x_best.get("PASS_constraint", False)) or bool(z_best.get("PASS_constraint", False))),
            "status": "PASS" if bool(bool(x_best.get("PASS_constraint", False)) or bool(z_best.get("PASS_constraint", False))) else "DIAG",
        })
    except Exception as exc:
        rows.append(error_row("13_feedback_stability", case.graph_id, exc, "K_CASAL_REDUCES_CONSTRAINT_BUT_HURTS_MATCH"))
df_13 = dataframe_result("13_feedback_stability", rows)
display(df_13)


,test,graph,what_it_tests,x_frac_rmse,z_frac_rmse,x_k_residual,z_k_residual,x_xz_residual,z_xz_residual,x_volume_ratio,z_volume_ratio,PASS,status
0,13_feedback_stability,1,x-feedback versus z-feedback stability compari...,0.281385,0.263100,5.000239e-07,1.056278e-07,0.415634,0.095075,0.670033,0.082918,True,PASS
1,13_feedback_stability,2,x-feedback versus z-feedback stability compari...,0.185704,0.190767,3.503420e-07,1.183037e-07,0.092309,0.102930,1.095763,1.012529,True,PASS
2,13_feedback_stability,3,x-feedback versus z-feedback stability compari...,0.203011,0.162919,1.269618e-07,1.386322e-07,0.418237,0.072047,1.005439,1.147129,True,PASS


## Final readiness

In [ ]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

KSPACE_MAP_VALID = bool(df_2["PASS"].fillna(False).all() and df_3["PASS"].fillna(False).all()) if ("df_2" in globals() and "df_3" in globals()) else False
GT_K_CONSTRAINT_VALID = bool(df_4["PASS"].fillna(False).all()) if "df_4" in globals() else False
GT_LATTICE_ORACLE_USEFUL = bool((df_6_pair["delta_match_vs_baseline"].mean() > 0.0) or (df_6_pair["delta_frac_rmse_vs_baseline"].mean() < 0.0)) if "df_6_pair" in globals() and not df_6_pair.empty else False
KSPACE_CASAL_STABLE = bool(df_8["PASS"].fillna(False).all() and df_9[df_9["method"].astype(str).str.startswith("late_k_casal_")]["PASS_constraint"].fillna(False).any()) if ("df_8" in globals() and "df_9" in globals()) else False
KSPACE_CASAL_KLDM_SAFE = bool(df_12["PASS"].fillna(False).all()) if "df_12" in globals() else False
KSPACE_CASAL_IMPROVES_FULL_WYCKOFF = bool(df_10[df_10["method"].eq("LateKSpaceCASAL_plus_full_Wyckoff")]["PASS_csp"].fillna(False).any()) if "df_10" in globals() and not df_10.empty else False
USE_AS_DEFAULT = bool(GT_K_CONSTRAINT_VALID and KSPACE_CASAL_STABLE and KSPACE_CASAL_KLDM_SAFE and KSPACE_CASAL_IMPROVES_FULL_WYCKOFF)
USE_AS_RESCUE_BRANCH = bool(GT_K_CONSTRAINT_VALID and KSPACE_CASAL_STABLE and not USE_AS_DEFAULT and KSPACE_CASAL_IMPROVES_FULL_WYCKOFF)

print(f"KSPACE_MAP_VALID={KSPACE_MAP_VALID}")
print(f"GT_K_CONSTRAINT_VALID={GT_K_CONSTRAINT_VALID}")
print(f"GT_LATTICE_ORACLE_USEFUL={GT_LATTICE_ORACLE_USEFUL}")
print(f"KSPACE_CASAL_STABLE={KSPACE_CASAL_STABLE}")
print(f"KSPACE_CASAL_KLDM_SAFE={KSPACE_CASAL_KLDM_SAFE}")
print(f"KSPACE_CASAL_IMPROVES_FULL_WYCKOFF={KSPACE_CASAL_IMPROVES_FULL_WYCKOFF}")
print(f"USE_AS_DEFAULT={USE_AS_DEFAULT}")
print(f"USE_AS_RESCUE_BRANCH={USE_AS_RESCUE_BRANCH}")


,test_name,rows,pass_count,error_count
0,0_imports_and_seed,3,3,0
1,1_l_roundtrip,9,9,0
2,2_k_metric_roundtrip,9,9,0
3,3_k_map_compare,9,9,0
4,4_gt_k_residual,10,10,0
5,5_family_projection,6,6,0
6,6_oracle_bottleneck,50,50,0
7,7_one_step_k_casal,3,2,0
8,8_synthetic_k_casal,9,9,0
9,9_late_k_casal,18,17,0


KSPACE_MAP_VALID=True
GT_K_CONSTRAINT_VALID=True
FINAL_K_PROJECTION_USEFUL=True
KSPACE_CASAL_STABLE=True
KSPACE_CASAL_KLDM_SAFE=True
KSPACE_CASAL_IMPROVES_FULL_WYCKOFF=True
USE_AS_DEFAULT=True
USE_AS_RESCUE_BRANCH=False


/tmp/ipykernel_22811/582112472.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  KSPACE_CASAL_STABLE = bool(df_8["PASS"].fillna(False).all() and df_9[df_9["method"].astype(str).str.startswith("late_k_casal_")]["PASS_constraint"].fillna(False).any()) if ("df_8" in globals() and "df_9" in globals()) else False
